In [5]:
# import path
import sys
sys.path.append("../src/")
sys.path.append("../")
from segment2d import *
import numpy as np
import csv
from matplotlib import pyplot as plt
from ipywidgets import interact
# visualize the image and mask in z ax is using interact, image and mask are in one slice
import SimpleITK as sitk
import scipy.ndimage as ndimage
from metrics_segmentation import hd
from natsort import natsorted
import configparser
from pathlib import Path
import pandas as pd
import os


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes= 5
model = FCDenseNet(in_channels=1, n_classes=num_classes)
class_weight = [0.27, 10.00, 8.91, 8.94]  #
segmenter = Segmenter_ACDC(
    model,
    class_weight,
    num_classes,
    0.001,
    0.5,
    50,
)
segmenter.eval()

checkpoint = "weights_EMIDEC_train_full/dice_0.7532.ckpt"
segmenter = Segmenter_ACDC.load_from_checkpoint(
    checkpoint_path=checkpoint,
    model=model,
    class_weight=class_weight,
    num_classes=num_classes,
    learning_rate=0.001,
    factor_lr=0.5,
    patience_lr=50,
)
segmenter = segmenter.to(device)

FileNotFoundError: [Errno 2] No such file or directory: '/home/nhattm/EMIDEC/notebooks/weights_EMIDEC_train_full/dice_0.7532.ckpt'

In [ ]:
model_emidec = FCDenseNet(in_channels=1, n_classes=5)

def export_torchscript_fp16(
    model,
    checkpoint_path,
    save_path="../saved_models/weights_EMIDEC_train_full/fcdensenet_fp16_scripted.pt",
    input_shape=(1, 1, 256, 256),
    device="cuda",
):
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.to(device)
    model.eval()
    model.half()

    example_input = torch.randn(*input_shape, device=device).half()

    with torch.no_grad():
        traced = torch.jit.trace(model, example_input)

    traced.save(save_path)
    print(f"Saved TorchScript FP16 model to: {save_path}")

    return traced
scripted_fp16 = export_torchscript_fp16(
    model_emidec,
    checkpoint_path="../saved_models/weights_EMIDEC_train_full/weight_tiramisu_emidec.pt",
    save_path="../saved_models/weights_EMIDEC_train_full/tiramisu_emidec_fp16.pt",
    input_shape=(1, 1, 256, 256),
    device="cuda",
)

Saved TorchScript FP16 model to: ../saved_models/weights_EMIDEC_train_full/tiramisu_emidec_fp16.pt


In [8]:
import torch
import torch.onnx
import sys
sys.path.append("../src/")
from segment2d import FCDenseNet
model_emidec = FCDenseNet(in_channels=1, n_classes=5)
def export_to_onnx(
    model,
    checkpoint_path,
    save_path="../saved_models/weights_EMIDEC_train_full/fcdensenet.onnx",
    input_shape=(1, 1, 256, 256),
    device="cpu",
):
    # Load the model weights
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.to(device)
    model.eval()

    # Prepare a dummy input to trace the model
    example_input = torch.randn(*input_shape, device=device)

    # Export the model to ONNX format
    torch.onnx.export(
        model,  # model to be exported
        example_input,  # example input
        save_path,  # path to save the model
        export_params=True,  # store the trained parameters in the model file
        opset_version=12,  # ONNX opset version (use version 12 for better compatibility)
        do_constant_folding=True,  # optimize the constant folding
        input_names=["input"],  # input tensor name
        output_names=["output"],  # output tensor name
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},  # dynamic batch size
    )

    print(f"Saved ONNX model to: {save_path}")

# Call the function to export your model to ONNX
export_to_onnx(
    model_emidec,
    checkpoint_path="../saved_models/weights_EMIDEC_train_full/weight_tiramisu_emidec.pt",
    save_path="../saved_models/weights_EMIDEC_train_full/tiramisu_emidec_fp32.onnx",
    input_shape=(1, 1, 256, 256),
    device="cuda",
)

Saved ONNX model to: ../saved_models/weights_EMIDEC_train_full/tiramisu_emidec_fp32.onnx


In [3]:

import onnxruntime as ort
import numpy as np

# ort.set_default_logger_severity(0)  # Suppress ONNX Runtime warnings

def predict_patches_onnx_gpu(images, onnx_model_path, num_classes=4, batch_size=8, fp16=False, device="cuda"):
    """Returns the patches using ONNX Runtime on GPU"""
    
    # Set execution providers for GPU (CUDA)
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]  # Use CUDA for GPU, fallback to CPU

    # Load the ONNX model with GPU support
    session = ort.InferenceSession(onnx_model_path, providers=providers)
    
    # Prepare an empty array for predictions
    prediction = np.zeros(
        (images.shape[0], num_classes, images.shape[2], images.shape[3]), dtype=np.float32
    )

    batch_start = 0
    batch_end = batch_size
    while batch_start < images.shape[0]:
        image = images[batch_start:batch_end]
        
        # Convert the image batch to float32 (ONNX Runtime expects this format)
        image = image.astype(np.float32)
        
        # If fp16 is enabled, convert to FP16
        if fp16:
            image = image.astype(np.float16)
        
        # Run inference with ONNX Runtime
        inputs = {session.get_inputs()[0].name: image}
        y_pred = session.run(None, inputs)

        # Store the prediction result
        prediction[batch_start:batch_end] = y_pred[0]

        batch_start += batch_size
        batch_end += batch_size

    return prediction

# Example usage of the ONNX-based prediction function:
# Assuming `images` is a numpy array with shape (batch_size, channels, height, width)
# and `onnx_model_path` is the path to the saved ONNX model

onnx_model_path = "../saved_models/weights_EMIDEC_train_full/tiramisu_emidec_fp32.onnx"
images = np.random.randn(8, 1, 256, 256).astype(np.float32)  # Example input
predictions = predict_patches_onnx_gpu(images, onnx_model_path, num_classes=5, batch_size=8, fp16=False)

# print(predictions)

# predict emidec


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open("csv_files/EMIDEC_test_train_full.csv", mode="r") as f:
    reader = csv.DictReader(f)
    list_test_subject = [row["path"] for row in reader]

model_emidec = torch.load("tiramisu_emidec.pt", weights_only=False)
model_emidec.eval()
model_emidec = model_emidec.to(device)

In [ ]:
import time
import torch
import pynvml  # nvidia-ml-py

def measure_energy_nvml(model, input_tensor, device='cuda:0', sample_interval=0.01, num_warmup=3):
    """
    Measure energy consumption (Joules) for a PyTorch operation using nvidia-ml-py.
    
    Args:
        model: PyTorch model (nn.Module).
        input_tensor: Input tensor (e.g., shape (1, C, H, W) for a single patient prediction).
        device: CUDA device (e.g., 'cuda:0').
        sample_interval: Time between power samples (seconds; default 10ms for accuracy).
        num_warmup: Number of dry runs to stabilize GPU (default 3).
    
    Returns:
        dict: {'energy_joules': float, 'avg_power_w': float, 'inference_time_s': float}
    """
    # Initialize NVML
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(int(device.split(':')[-1]))
    
    # Warmup runs to stabilize GPU clocks
    model.eval().to(device)
    input_tensor = input_tensor.to(device)
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(input_tensor)
    torch.cuda.synchronize()  # Ensure warmup completes
    
    # Power sampling during inference
    power_samples = []
    start_time = time.time()
    
    with torch.no_grad():
        torch.cuda.synchronize()  # Sync before start
        start_sample = time.time()
        output = model(input_tensor)
        torch.cuda.synchronize()  # Sync after end
        end_sample = time.time()
    
    inference_time = end_sample - start_sample
    
    # Sample power during the inference window (high-frequency polling)
    sample_start = start_time
    while sample_start < end_sample:
        power_mw = pynvml.nvmlDeviceGetPowerUsage(handle)
        power_samples.append(power_mw / 1000.0)  # Convert to Watts
        time.sleep(sample_interval)
        sample_start += sample_interval
    
    avg_power = sum(power_samples) / len(power_samples) if power_samples else 0.0
    energy_joules = avg_power * inference_time
    
    # Cleanup
    pynvml.nvmlShutdown()
    
    return {
        'energy_joules': energy_joules,
        'avg_power_w': avg_power,
        'inference_time_s': inference_time
    }
    
patient_input = torch.randn(1, 1, 256, 256)  # Example: single-channel 256x256 MRI slice

# Measure
results = measure_energy_nvml(model_emidec, patient_input)
print(f"Energy per patient prediction: {results['energy_joules']:.4f} J")
print(f"Average power: {results['avg_power_w']:.2f} W")
print(f"Inference time: {results['inference_time_s']:.4f} s")

In [ ]:
import pynvml

metrics_emidec_header = ["patient_name", 
    "Dice_Myocardium","HD_Myocardium", "Volume_Myocardium", "Err_Myocardium(ml)",
    "Dice_Infarction", "Volume_Infarction", "Err_Infarction(ml)", "Vol_Difference_Infarction_rate(%)",
    "Dice_No-Reflow", "Volume_No-Reflow", "Err_No-Reflow(ml)", "Vol_Difference_No-Reflow_rate(%)",
    ]
# create folder to save the predicted image
os.makedirs("predicted_emidec", exist_ok=True)

with open("csv_files/EMIDEC_test_train_full.csv", mode="r") as f:
    reader = csv.DictReader(f)
    list_test_subject = [row["path"] for row in reader]
# open csv file to save the result
f = open("csv_files/result_emidec.csv", mode="w")
writer = csv.DictWriter(f, fieldnames=metrics_emidec_header)
writer.writeheader()

#open csv file to save the volume of GT
f_volume = open("csv_files/volume_emidec_GT.csv", mode="w")
writer_volume = csv.DictWriter(f_volume, fieldnames=["patient_name", "Volume_Myocardium", "Volume_Infarction", "Volume_No-Reflow"])
writer_volume.writeheader()

for image_path in list_test_subject:

    patient_name = image_path.split("/")[-3]
    # print("patient name: ", patient_name)
    mask_path = image_path.replace("Images", "Contours")
    image, affine, header = load_nii(image_path)
    mask, _, _ = load_nii(mask_path)
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    data = preprocess_data_nii(image_path)
    seg = predict_data_model_emidec(data, model_emidec, min_size_remove=500).astype(np.uint8)
    power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000  # watts
    pynvml.nvmlShutdown()
    print("power: ", power)
    save_nii(f"predicted_emidec/{patient_name}_tiramisu_seg.nii.gz", seg, affine, header)

    result = metrics_EMIDEC(mask, seg, voxel_size=header.get_zooms())
    # round the result to 3 decimal places
    result = [patient_name] + [round(r, 4) for r in result]
    # write the result to the csv file
    writer.writerow(dict(zip(metrics_emidec_header, result)))
    # break
    #save the volume of GT
    mask_myo = (mask == 2) + (mask == 3) + (mask == 4)
    mask_infarction = (mask == 3) + (mask == 4)
    mask_noreflow = (mask == 4)
    volume_myo = mask_myo.sum() * np.prod(header.get_zooms()) / 1000.
    volume_infarction = mask_infarction.sum() * np.prod(header.get_zooms()) / 1000.
    volume_noreflow = mask_noreflow.sum() * np.prod(header.get_zooms()) / 1000.
    writer_volume.writerow({"patient_name": patient_name, "Volume_Myocardium": volume_myo, "Volume_Infarction": volume_infarction, "Volume_No-Reflow": volume_noreflow})
f.close()
f_volume.close()

In [ ]:
# crop the image and mask to the center of the image
image_show = image[image.shape[0]//2-64:image.shape[0]//2+64, image.shape[1]//2-64:image.shape[1]//2+64, :]
mask_show = mask[mask.shape[0]//2-64:mask.shape[0]//2+64, mask.shape[1]//2-64:mask.shape[1]//2+64, :]
seg_show = seg[seg.shape[0]//2-64:seg.shape[0]//2+64, seg.shape[1]//2-64:seg.shape[1]//2+64, :]

def plot_image_mask_z(image, mask, padded_image, padded_mask, z):
    fig, ax = plt.subplots(1, 3, figsize=(10, 5))
    ax[0].imshow(image[..., z], cmap="gray")
    ax[0].set_title("Image gt")
    # ax[0].imshow(mask[..., z], cmap="jet", alpha=0.3)
    ax[1].imshow(padded_image[..., z], cmap="gray")
    ax[1].set_title("Image predict")
    ax[1].imshow(padded_mask[..., z], cmap="jet", alpha=0.5, vmin=0, vmax=4)

    ax[2].imshow(padded_image[..., z], cmap="gray", alpha=0.7)
    ax[2].imshow(mask[..., z], cmap="jet", alpha=0.5, vmin=0, vmax=4)
    ax[2].set_title("Mask gt")
    # off the axis
    ax[0].axis('off')
    ax[1].axis('off')
    ax[2].axis('off')
    # add legend
    ax[1].legend(["Myocardium", "Infarction", "No-Reflow"], loc="upper right")


interact(lambda z: plot_image_mask_z(image_show, mask_show, image_show, seg_show, z), z=(0, image.shape[-1] - 1))

# predict acdc

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# num_classes= 4
# model = FCDenseNet(in_channels=1, n_classes=num_classes)
# class_weight = [0.27, 10.00, 8.91, 8.94]  #
# segmenter = Segmenter_ACDC(
#     model,
#     class_weight,
#     num_classes,
#     0.001,
#     0.5,
#     50,
# )
# segmenter.eval()

# checkpoint = "weights_ACDC/dice_0.9137.ckpt"
# segmenter = Segmenter_ACDC.load_from_checkpoint(
#     checkpoint_path=checkpoint,
#     model=model,
#     class_weight=class_weight,
#     num_classes=num_classes,
#     learning_rate=0.001,
#     factor_lr=0.5,
#     patience_lr=50,
# )
# segmenter = segmenter.to(device)
model_acdc = torch.load("../saved_models/tiramisu_acdc.pt", weights_only=False)
model_acdc.eval()
model_acdc = model_acdc.to(device)

In [ ]:
model_acdc = FCDenseNet(in_channels=1, n_classes=4)

def export_torchscript_fp16(
    model,
    checkpoint_path,
    save_path="../saved_models/weights_ACDC/fcdensenet_fp16_scripted.pt",
    input_shape=(1, 1, 256, 256),
    device="cuda",
):
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.to(device)
    model.eval()
    model.half()

    example_input = torch.randn(*input_shape, device=device).half()

    with torch.no_grad():
        traced = torch.jit.trace(model, example_input)

    traced.save(save_path)
    print(f"Saved TorchScript FP16 model to: {save_path}")

    return traced
scripted_fp16 = export_torchscript_fp16(
    model_acdc,
    checkpoint_path="../saved_models/weights_ACDC/weight_tiramisu_acdc.pt",
    save_path="../saved_models/weights_ACDC/tiramisu_acdc_fp16.pt",
    input_shape=(1, 1, 256, 256),
    device="cuda",
)

Saved TorchScript FP16 model to: ../saved_models/weights_ACDC/tiramisu_acdc_fp16.pt


In [6]:
import torch
import torch.onnx
model_emidec = FCDenseNet(in_channels=1, n_classes=4)
def export_to_onnx(
    model,
    checkpoint_path,
    save_path="../saved_models/weights_ACDC/weight_tiramisu_acdc.pt",
    input_shape=(1, 1, 256, 256),
    device="cpu",
):
    # Load the model weights
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.to(device)
    model.eval()

    # Prepare a dummy input to trace the model
    example_input = torch.randn(*input_shape, device=device)

    # Export the model to ONNX format
    torch.onnx.export(
        model,  # model to be exported
        example_input,  # example input
        save_path,  # path to save the model
        export_params=True,  # store the trained parameters in the model file
        opset_version=12,  # ONNX opset version (use version 12 for better compatibility)
        do_constant_folding=True,  # optimize the constant folding
        input_names=["input"],  # input tensor name
        output_names=["output"],  # output tensor name
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},  # dynamic batch size
    )

    print(f"Saved ONNX model to: {save_path}")

# Call the function to export your model to ONNX
export_to_onnx(
    model_emidec,
    checkpoint_path="../saved_models/weights_ACDC/weight_tiramisu_acdc.pt",
    save_path="../saved_models/weights_ACDC/tiramisu_acdc_fp32.onnx",
    input_shape=(1, 1, 256, 256),
    device="cuda",
)

Saved ONNX model to: ../saved_models/weights_ACDC/tiramisu_acdc_fp32.onnx


In [1]:
import onnxruntime as ort
import numpy as np
# ort.set_default_logger_severity(0) 



def predict_patches_onnx_gpu(images, onnx_model_path, num_classes=4, batch_size=8, fp16=False, device="cuda"):
    """Returns the patches using ONNX Runtime on GPU"""
    
    # Set execution providers for GPU (CUDA)
    # providers = ['CUDAExecutionProvider']  # Use CUDA for GPU
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(onnx_model_path, providers=providers)




    # Prepare an empty array for predictions
    prediction = np.zeros(
        (images.shape[0], num_classes, images.shape[2], images.shape[3]), dtype=np.float32
    )

    batch_start = 0
    batch_end = batch_size
    while batch_start < images.shape[0]:
        image = images[batch_start:batch_end]
        
        # Convert the image batch to float32 (ONNX Runtime expects this format)
        image = image.astype(np.float32)
        
        # If fp16 is enabled, convert to FP16
        if fp16:
            image = image.astype(np.float16)
        
        # Run inference with ONNX Runtime
        inputs = {session.get_inputs()[0].name: image}
        y_pred = session.run(None, inputs)

        # Store the prediction result
        prediction[batch_start:batch_end] = y_pred[0]

        batch_start += batch_size
        batch_end += batch_size

    return prediction

# Example usage of the ONNX-based prediction function:
# Assuming `images` is a numpy array with shape (batch_size, channels, height, width)
# and `onnx_model_path` is the path to the saved ONNX model

onnx_model_path = "../saved_models/weights_ACDC/tiramisu_acdc_fp32.onnx"
images = np.random.randn(8, 1, 256, 256).astype(np.float32)  # Example input
predictions = predict_patches_onnx_gpu(images, onnx_model_path, num_classes=4, batch_size=8, fp16=False)

# print(predictions)

In [ ]:
metrics_acdc_header = ["patient_name", 
            "Dice ED Left Ventricle", "HD ED Left Ventricle", "Volume ED Left Ventricle", "Err ED Left Ventricle(ml)",
          "Dice ED Right Ventricle", "HD ED Right Ventricle", "Volume ED Right Ventricle", "Err ED Right Ventricle(ml)",
          "Dice ED Myocardium", "HD ED Myocardium", "Volume ED Myocardium", "Err ED Myocardium(ml)",
          "Dice ES Left Ventricle", "HD ES Left Ventricle", "Volume ES Left Ventricle", "Err ES Left Ventricle(ml)",
          "Dice ES Right Ventricle", "HD ES Right Ventricle", "Volume ES Right Ventricle", "Err ES Right Ventricle(ml)",
          "Dice ES Myocardium", "HD ES Myocardium", "Volume ES Myocardium", "Err ES Myocardium(ml)"]

config = configparser.ConfigParser()
with open("../data/csv_files/ACDC_test.csv", mode="r") as f:
    reader = csv.DictReader(f)
    list_test_subject = [row["path"] for row in reader]

test_patitent_paths = natsorted(glob.glob("../data/ACDC/database/testing/patient*"))


# open csv file to save the result
f = open("../data/csv_files/result_acdc.csv", mode="w")
writer = csv.DictWriter(f, fieldnames=metrics_acdc_header)
writer.writeheader()
# create folder to save the predicted mask
os.makedirs("../data/predicted_acdc", exist_ok=True)
fb_16 = True
device = torch.device("cpu")
for patient_path in test_patitent_paths:

    info_path = patient_path + "/Info.cfg"
    with open(info_path, mode="r") as f:
        config.read_string(f"[info]\n{f.read()}")
    ED_index = f"0{config['info']['ED']}" if int(config['info']['ED']) < 10 else f"{config['info']['ED']}"
    ES_index = f"0{config['info']['ES']}" if int(config['info']['ES']) < 10 else f"{config['info']['ES']}"
    patient_name = patient_path.split("/")[-1]
    ED_image_path = patient_path + f"/{patient_name}_frame{ED_index}.nii.gz"
    ES_image_path = patient_path + f"/{patient_name}_frame{ES_index}.nii.gz"
    ED_mask_path = patient_path + f"/{patient_name}_frame{ED_index}_gt.nii.gz"
    ES_mask_path = patient_path + f"/{patient_name}_frame{ES_index}_gt.nii.gz"


    ED_image, ED_affine, ED_header = load_nii(ED_image_path)
    ES_image, ES_affine, ES_header = load_nii(ES_image_path)
    ED_mask, _, _ = load_nii(ED_mask_path)
    ES_mask, _, _ = load_nii(ES_mask_path)

    ED_data = preprocess_data_nii(ED_image_path)
    ES_data = preprocess_data_nii(ES_image_path)
    ED_seg = predict_data_model(ED_data, model_acdc, min_size_remove=800, fp16=fb_16, device=device).astype(np.uint8)
    ES_seg = predict_data_model(ES_data, model_acdc, min_size_remove=800, fp16=fb_16, device=device).astype(np.uint8)

    result_ED = metrics_ACDC(ED_mask, ED_seg, voxel_size=ED_header.get_zooms())
    result_ES = metrics_ACDC(ES_mask, ES_seg, voxel_size=ES_header.get_zooms())
    # round the result to 3 decimal places
    result = [patient_name] + [round(r, 4) for r in result_ED] + [round(r, 4) for r in result_ES]

    # write the result to the csv file
    writer.writerow(dict(zip(metrics_acdc_header, result)))
    # save the predicted mask
    save_nii(f"../data/predicted_acdc/{patient_name}_tiramisu_seg_ED.nii.gz", ED_seg, ED_affine, ED_header)
    save_nii(f"../data/predicted_acdc/{patient_name}_tiramisu_seg_ES.nii.gz", ES_seg, ES_affine, ES_header)
    # break

f.close()

In [ ]:
import pandas as pd
# read the result_acdc.csv file
result_acdc = pd.read_csv("../data/csv_files/result_acdc.csv")
# remove the Volume column
result_acdc = result_acdc.drop(columns=["Volume ED Left Ventricle", "Volume ED Right Ventricle", "Volume ED Myocardium", "Volume ES Left Ventricle", "Volume ES Right Ventricle", "Volume ES Myocardium"])
result_acdc.head()
# Melt everything except patient_name
df_long = result_acdc.melt(id_vars='patient_name', var_name='Metric_Label', value_name='Value')

# Extract Metric type (Dice, HD, Err), Phase (ED, ES), and Region (LV, RV, MYO)
df_long[['Metric', 'Phase', 'Region']] = df_long['Metric_Label'].str.extract(r'(\w+)\s+(ED|ES)\s+(Left Ventricle|Right Ventricle|Myocardium)')

result_emidec = pd.read_csv("../data/csv_files/result_emidec.csv")
result_emidec = result_emidec.drop(columns=["Volume_Myocardium","Volume_Infarction", "Volume_No-Reflow","HD_Myocardium"])
result_emidec.head()
df_emidec = result_emidec.melt(id_vars='patient_name', 
                               var_name='Metric_Label', 
                               value_name='Value')

# Extract Metric + Region
df_emidec[['Metric', 'Region']] = df_emidec['Metric_Label'].str.extract(
    r'(Dice|Err|Vol_Difference).*_(Myocardium|Infarction|No-Reflow)'
)

# Rename 'Err' to 'Volume Error' for clarity
df_emidec['Metric'] = df_emidec['Metric'].replace({'Err': 'Volume Error'})

# remove all row that have Metric_Label == Dice_Infarction and Value == 0 and == 1
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Dice_Infarction') & (df_emidec['Value'] == 0))]
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Dice_Infarction') & (df_emidec['Value'] == 1))]
# remove all row that have Metric_Label == Dice_No-Reflow and Value == 0 and == 1
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Dice_No-Reflow') & (df_emidec['Value'] == 0))]
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Dice_No-Reflow') & (df_emidec['Value'] == 1))]

# remove all row that have Metric_Label == Vol_Difference_Infarction_rate(%) and Value == 0
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Vol_Difference_Infarction_rate(%)') & (df_emidec['Value'] == 0))]
# remove all row that have Metric_Label == Vol_Difference_No-Reflow_rate(%) and Value == 0
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Vol_Difference_No-Reflow_rate(%)') & (df_emidec['Value'] == 0))]

# remove all row that have Metric_Label == Vol_Difference(mL) and Value == 0
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Err_Infarction(ml)') & (df_emidec['Value'] == 0))]
df_emidec = df_emidec[~((df_emidec['Metric_Label'] == 'Err_No-Reflow(ml)') & (df_emidec['Value'] == 0))]
# list unique Metric_Label
df_emidec['Metric_Label'].unique()

In [ ]:
result_emidec = pd.read_csv("../data/csv_files/result_emidec.csv")
gt_volume = pd.read_csv("../data/csv_files/volume_emidec_GT.csv")

In [ ]:
import numpy as np
from scipy import stats
import pandas as pd
gt = gt_volume["Volume_Myocardium"]
pred = result_emidec["Volume_Myocardium"]
# Your data
# get all [gt_volume["Volume_Infarction"] != 0] and divide by gt_volume["Volume_Myocardium"]
gt = gt_volume[gt_volume["Volume_Infarction"] != 0]
gt = gt["Volume_Infarction"] / gt["Volume_Myocardium"]
pred = result_emidec[result_emidec["Volume_Infarction"]!=0]
pred = pred["Volume_Infarction"] / pred["Volume_Myocardium"]

# Convert to numpy arrays
gt = np.array(gt)
pred = np.array(pred)

# Check same length
if len(gt) != len(pred):
    raise ValueError("Both volume arrays must have the same length for paired test.")

# Remove NaN values (Wilcoxon requires complete pairs)
valid_mask = ~(np.isnan(gt) | np.isnan(pred))
gt_clean = gt[valid_mask]
pred_clean = pred[valid_mask]

if len(gt_clean) < 10:
    print("Warning: Less than 10 valid pairs. Results may be unreliable.")

print(f"Number of valid pairs: {len(gt_clean)}")

# Perform Wilcoxon Signed-Rank Test
statistic, p_value = stats.wilcoxon(gt_clean, pred_clean)

# Output results
print(f"\nWilcoxon Signed-Rank Test results:")
print(f"Statistic: {statistic:.4f}")
print(f"p-value: {p_value:.4f}")

# Interpret significance
alpha = 0.05
if p_value < alpha:
    print("✅ Statistically significant difference (p < 0.05)")
else:
    print("❌ No statistically significant difference (p >= 0.05)")

# Additional statistics
differences = gt_clean - pred_clean
median_diff = np.median(differences)
mad = np.median(np.abs(differences - median_diff))  # Median Absolute Deviation

print(f"\nDescriptive statistics:")
print(f"Median difference (GT - Pred): {median_diff:.2f}")
print(f"MAD (Median Absolute Deviation): {mad:.2f}")
print(f"Mean difference: {np.mean(differences):.2f}")
print(f"Std difference: {np.std(differences, ddof=1):.2f}")

# Effect size (r = z / sqrt(N))
n = len(gt_clean)
z_stat = abs(stats.norm.ppf(p_value / 2))  # Approximate z-score
effect_size_r = z_stat / np.sqrt(n)
print(f"Effect size (r): {effect_size_r:.3f}")
print(f"  - 0.1 = small, 0.3 = medium, 0.5 = large")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Create a single figure with 2 rows × 3 columns
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# ==========================================================
# === Row 1: EMIDEC dataset
# ==========================================================
metrics_emidec = ['Dice', 'Vol_Difference', 'Volume Error']
titles_emidec = ['Dice Score', 'Volume Difference Rate(%)', 'Volume Difference (mL)']

for ax, metric, title in zip(range(len(axes[0])), metrics_emidec, titles_emidec):
    if ax != 1:
        ax = axes[0][ax]
        subset = df_emidec[df_emidec['Metric'] == metric]
        
        sns.boxplot(
            x='Region',
            y='Value',
            hue='Region',
            data=subset,
            ax=ax,
            showfliers=True,  # optional: hides outliers for cleaner look
            legend=False
        )
        
        # ax.set_title(title)
        ax.set_xlabel('Region')
        ax.set_ylabel(metric)
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        
        if metric == 'Dice':
            ax.set_ylim(0, 1)
            ax.set_ylabel('Dice Score')
        elif metric == 'Vol_Difference':
            ax.set_ylim(0, None)
            ax.set_ylabel('Difference Rate(%)')
        elif metric == 'Volume Error':
            ax.set_ylim(0, None)
            ax.set_ylabel('Volume Difference (mL)')

# Add EMIDEC subtitle manually
fig.text(0.5, 0.96, 'Segmentation Metrics Across Regions on EMIDEC Dataset',
         ha='center', va='center', fontsize=14, fontweight='bold')
########################### plot the ax[0][1]
ax = axes[0][1]
gt = gt_volume[gt_volume["Volume_Infarction"] != 0]
gt_ratio = gt["Volume_Infarction"] / gt["Volume_Myocardium"] * 100

pred = result_emidec[result_emidec["Volume_Infarction"] != 0]
pred_ratio = pred["Volume_Infarction"] / pred["Volume_Myocardium"] * 100
# Ensure same indexing if needed (optional: align by patient ID)
# Here we assume they are already paired or independent for visualization

# Create a DataFrame for easy plotting with seaborn
data = pd.DataFrame({
    'AI': pred_ratio,
    'Ground Truth': gt_ratio,

})
data_melted = data.melt(var_name='', value_name='Infarction / Myocardium Ratio')
sns.boxplot(x='', y='Infarction / Myocardium Ratio', data=data_melted, palette="Set2", ax=ax)
# make the boxplot of gt_volume["Volume_Infarction"] / gt_volume["Volume_Myocardium"] and result_emidec["Volume_Infarction"] / result_emidec["Volume_Myocardium"]
# ax.set_title("Scar Burden")
ax.set_ylabel('Scar Burden (%)')



###############################################################
# ==========================================================
# === Row 2: ACDC dataset
# ==========================================================
metrics_acdc = ['Dice', 'HD', 'Err']
titles_acdc = ['Dice Score', 'Hausdorff Distance (mm)', 'Volume Difference (mL)']

for ax, metric, title in zip(axes[1], metrics_acdc, titles_acdc):
    subset = df_long[df_long['Metric'] == metric]
    
    sns.boxplot(
        x='Phase',
        y='Value',
        hue='Region',
        data=subset,
        ax=ax,
        showfliers=True,
        legend=True
    )
    
    # ax.set_title(title)
    ax.set_xlabel('Phase')
    ax.set_ylabel(metric)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    
    if metric == 'Dice':
        ax.set_ylim(0, 1)
        ax.set_ylabel('Dice Score')
    elif metric == 'HD':
        ax.set_ylim(0, None)
        ax.set_ylabel('Hausdorff Distance (mm)')
    elif metric == 'Err':
        ax.set_ylabel('Volume Difference (mL)')

plt.subplots_adjust(hspace=0.9, bottom=0.12)

# Add ACDC subtitle manually
fig.text(0.5, 0.47, 'Segmentation Metrics Across End-Diastolic (ED) and End-Systolic (ES) Phases on ACDC Dataset',
         ha='center', va='center', fontsize=14, fontweight='bold')

# ==========================================================
# === Final formatting
# ==========================================================

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.subplots_adjust(hspace=0.25)
plt.savefig("../figures/segmentation_metrics.png", dpi=300)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Create a single figure with 2 rows × 3 columns
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# ==========================================================
# === Row 1: EMIDEC dataset
# ==========================================================
metrics_emidec = ['Dice', 'Vol_Difference', 'Volume Error']
titles_emidec = ['Dice Score', 'Volume Difference Rate(%)', 'Volume Difference (mL)']

for ax, metric, title in zip(range(len(axes[0])), metrics_emidec, titles_emidec):
    if ax != 1:
        ax = axes[0][ax]
        subset = df_emidec[df_emidec['Metric'] == metric]
        
        sns.boxplot(
            x='Region',
            y='Value',
            hue='Region',
            data=subset,
            ax=ax,
            showfliers=True,  # optional: hides outliers for cleaner look
            legend=False
        )
        
        # ax.set_title(title)
        ax.set_xlabel('Region')
        ax.set_ylabel(metric)
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        
        if metric == 'Dice':
            ax.set_ylim(0, 1)
            ax.set_ylabel('Dice Score')
        elif metric == 'Vol_Difference':
            ax.set_ylim(0, None)
            ax.set_ylabel('Difference Rate(%)')
        elif metric == 'Volume Error':
            ax.set_ylim(0, None)
            ax.set_ylabel('Volume Difference (mL)')

# Add EMIDEC subtitle manually
fig.text(0.5, 0.96, 'Segmentation Metrics Across Regions on EMIDEC Dataset',
         ha='center', va='center', fontsize=14, fontweight='bold')
########################### plot the ax[0][1]
ax = axes[0][1]
gt = gt_volume[gt_volume["Volume_Infarction"] != 0]
gt_ratio = gt["Volume_Infarction"] / gt["Volume_Myocardium"] * 100

pred = result_emidec[result_emidec["Volume_Infarction"] != 0]
pred_ratio = pred["Volume_Infarction"] / pred["Volume_Myocardium"] * 100
# Ensure same indexing if needed (optional: align by patient ID)
# Here we assume they are already paired or independent for visualization

# Create a DataFrame for easy plotting with seaborn
data = pd.DataFrame({
    'AI': pred_ratio,
    'Ground Truth': gt_ratio,

})
data_melted = data.melt(var_name='', value_name='Infarction / Myocardium Ratio')
sns.boxplot(x='', y='Infarction / Myocardium Ratio', data=data_melted, palette="Set2", ax=ax)
# make the boxplot of gt_volume["Volume_Infarction"] / gt_volume["Volume_Myocardium"] and result_emidec["Volume_Infarction"] / result_emidec["Volume_Myocardium"]
# ax.set_title("Scar Burden")
ax.set_ylabel('Scar Burden (%)')



###############################################################
# ==========================================================
# === Row 2: ACDC dataset
# ==========================================================
metrics_acdc = ['Dice', 'HD', 'Err']
titles_acdc = ['Dice Score', 'Hausdorff Distance (mm)', 'Volume Difference (mL)']

for ax, metric, title in zip(axes[1], metrics_acdc, titles_acdc):
    subset = df_long[df_long['Metric'] == metric]
    
    sns.boxplot(
        x='Phase',
        y='Value',
        hue='Region',
        data=subset,
        ax=ax,
        showfliers=True,
        legend=True
    )
    
    # ax.set_title(title)
    ax.set_xlabel('Phase')
    ax.set_ylabel(metric)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    
    if metric == 'Dice':
        ax.set_ylim(0, 1)
        ax.set_ylabel('Dice Score')
    elif metric == 'HD':
        ax.set_ylim(0, None)
        ax.set_ylabel('Hausdorff Distance (mm)')
    elif metric == 'Err':
        ax.set_ylabel('Volume Difference (mL)')

plt.subplots_adjust(hspace=0.9, bottom=0.12)

# Add ACDC subtitle manually
fig.text(0.5, 0.47, 'Segmentation Metrics Across End-Diastolic (ED) and End-Systolic (ES) Phases on ACDC Dataset',
         ha='center', va='center', fontsize=14, fontweight='bold')

# ==========================================================
# === Final formatting
# ==========================================================

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.subplots_adjust(hspace=0.25)
plt.savefig("../figures/segmentation_metrics.png", dpi=300)

plt.show()


In [ ]:
# make the crop center of the image 128
image = ED_image[ED_image.shape[0]//2-64:ED_image.shape[0]//2+64, ED_image.shape[1]//2-64:ED_image.shape[1]//2+64, :]
mask = ED_mask[ED_mask.shape[0]//2-64:ED_mask.shape[0]//2+64, ED_mask.shape[1]//2-64:ED_mask.shape[1]//2+64, :]
seg = ED_seg[ED_seg.shape[0]//2-64:ED_seg.shape[0]//2+64, ED_seg.shape[1]//2-64:ED_seg.shape[1]//2+64, :]

def plot_image_mask_z(image, mask, padded_image, padded_mask, z):
    fig, ax = plt.subplots(1, 3, figsize=(10, 5))
    ax[0].imshow(image[..., z], cmap="gray")
    ax[0].set_title("Image gt")
    # ax[0].imshow(mask[..., z], cmap="jet", alpha=0.3)
    ax[1].imshow(padded_image[..., z], cmap="gray", alpha=0.7)
    ax[1].set_title("Image predict")
    ax[1].imshow(padded_mask[..., z], cmap="jet", alpha=0.5, vmin=0, vmax=4)

    ax[2].imshow(padded_image[..., z], cmap="gray", alpha=0.7)
    ax[2].imshow(mask[..., z], cmap="jet", alpha=0.5)
    ax[2].set_title("Mask gt")
    # off the axis
    ax[0].axis('off')
    ax[1].axis('off')


interact(lambda z: plot_image_mask_z(image, mask, image, seg, z), z=(0, ED_image.shape[-1] - 1))

# predict Table 2 dataset

In [ ]:
model_name = "tiramisu_emidec.pt" # "tiramisu_emidec.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_acdc = torch.load(model_name, weights_only=False)
model_acdc.eval()
model_acdc = model_acdc.to(device)

def preprocess_data_table2(image_nrrd_path):
    data = {}
    patient_image = nrrd.read(image_nrrd_path)
    image = patient_image[0]
    image = min_max_normalize(image)

    resized_image, restore_info = crop_resize_image(image, 256)
    # padded_mask = pad_background_with_index(mask, crop_index, padded_index, dim2pad=cfg.DATA.DIM2PAD)
    data["restore_info"] = restore_info
    batch_images = []
    for i in range(resized_image.shape[-1]):
        slice_inputs = resized_image[..., i : i + 1]  # shape (224, 224, 1)
        slices_image = torch.from_numpy(slice_inputs.transpose(-1, 0, 1))  # shape (1, 224, 224)
        batch_images.append(slices_image)

    batch_images = torch.stack(batch_images).float()  # shape (9,1, 224, 224)
    data["image"] = batch_images
    return data
    
table2_path = "Tables 2/Tables/*/"
list_test_image = natsorted([x for x in glob.glob("Tables 2/Tables/*/*.nrrd") if "seg" not in x])
list_test_mask = natsorted([x for x in glob.glob("Tables 2/Tables/*/*seg.nrrd")])

print(len(list_test_image), len(list_test_mask))


In [ ]:
# create folder to save the predicted masks
import nrrd
csv_file = f"csv_files/result_table2_{model_name}.csv"
f = open(csv_file, mode="w")
writer = csv.DictWriter(f, fieldnames=["patient_name", "Dice"])
writer.writeheader()
os.makedirs("predicted_table2_data_emidec_model", exist_ok=True)
os.makedirs("predicted_table2_data_acdc_model", exist_ok=True)

for index in range(len(list_test_image)):
    if "42_PC" in list_test_image[index]:

        image = nrrd.read(list_test_image[index])[0]
        image_info = nrrd.read(list_test_image[index])[1]
        data = preprocess_data_table2(list_test_image[index])

        # seg = predict_data(data, segmenter, patient=patient, mvo=is_MVO, task=task).astype(np.uint8)
        num_classes = 4 if model_name == "tiramisu_acdc.pt" else 5
        seg = predict_data_model(data, model_acdc, num_classes=num_classes).astype(np.uint8)
        # give label 1, 3, 4 to 0
        seg[seg == 1] = 0
        seg[seg == 3] = 0
        seg[seg == 4] = 0
        # give label 2 to 1
        seg[seg == 2] = 1
        # write the predicted mask to the a nrrd file with image_info
        # and make name of the file is like this 1 - PGF/34 de_high res PSIR EC_PSIR_tiramisu.nrrd
        name_predict = list_test_image[index].split("/")[-2:]
        name_predict = "/".join(name_predict).replace(".nrrd", "_tiramisu_seg.nrrd")
        # make folder dir 1-PGF to save the predicted mask
        if model_name == "tiramisu_acdc.pt":
            os.makedirs(f"predicted_table2_data_acdc_model/{name_predict.split('/')[0]}", exist_ok=True)
            nrrd.write(f"predicted_table2_data_acdc_model/{name_predict}", seg, image_info)
        else:
            os.makedirs(f"predicted_table2_data_emidec_model/{name_predict.split('/')[0]}", exist_ok=True)
            nrrd.write(f"predicted_table2_data_emidec_model/{name_predict}", seg, image_info)
            
        print("write predicted mask to ", name_predict)

        mask = nrrd.read(list_test_mask[index])[0]
        dice = dice_volume_ACDC(mask, seg, class_index=1)
        writer.writerow({"patient_name": list_test_image[index].split("/")[-2], "Dice": dice})

    
    # break
f.close()

In [ ]:
def plot_image_mask_z(image, mask, padded_image, padded_mask, z):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(image[..., z], cmap="gray")
    ax[0].set_title("Image gt")
    ax[0].imshow(mask[..., z], cmap="jet", alpha=0.3)
    ax[1].imshow(padded_image[..., z], cmap="gray")
    ax[1].set_title("Image predict")
    ax[1].imshow(padded_mask[..., z], cmap="jet", alpha=0.3)
    # off the axis
    ax[0].axis('off')
    ax[1].axis('off')


interact(lambda z: plot_image_mask_z(image, mask, image, seg, z), z=(0, image.shape[-1] - 1))

# predict mouse data


In [ ]:
from pydicom import dcmread
import pydicom
import torch
model_name = "../saved_models/tiramisu_acdc.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_acdc = torch.load(f"{model_name}", weights_only=False)
model_acdc.eval()
model_acdc = model_acdc.to(device)
def preprocess_data_mouse(image_dcm_path):
    data = {}
    subject = dcmread(image_dcm_path)
    image = subject.pixel_array.astype(np.float32)
    image = min_max_normalize(image)
    data["is_2D"] = False
    if len(image.shape) == 2:
        image = np.expand_dims(image, axis=-1)
        data["is_2D"] = True

    resized_image, restore_info = crop_resize_image(image, 256)
    # padded_mask = pad_background_with_index(mask, crop_index, padded_index, dim2pad=cfg.DATA.DIM2PAD)
    data["restore_info"] = restore_info
    batch_images = []
    for i in range(resized_image.shape[-1]):
        slice_inputs = resized_image[..., i : i + 1]  # shape (224, 224, 1)
        slices_image = torch.from_numpy(slice_inputs.transpose(-1, 0, 1))  # shape (1, 224, 224)
        batch_images.append(slices_image)

    batch_images = torch.stack(batch_images).float()  # shape (9,1, 224, 224)
    data["image"] = batch_images
    #save metadata
    data["subject"] = subject
    return data

def preprocess_data_mouse_retro(image_array):
    data = {}

    image = min_max_normalize(image_array)
    resized_image, restore_info = crop_resize_image(image, 256)
    # padded_mask = pad_background_with_index(mask, crop_index, padded_index, dim2pad=cfg.DATA.DIM2PAD)
    data["restore_info"] = restore_info
    batch_images = []
    for i in range(resized_image.shape[-1]):
        slice_inputs = resized_image[..., i : i + 1]  # shape (224, 224, 1)
        slices_image = torch.from_numpy(slice_inputs.transpose(-1, 0, 1))  # shape (1, 224, 224)
        batch_images.append(slices_image)

    batch_images = torch.stack(batch_images).float()  # shape (9,1, 224, 224)
    data["image"] = batch_images

    return data 
    
from pydicom.uid import generate_uid


def create_segmentation_dicom(original_dcm: pydicom.dataset.FileDataset,
                              seg_array: np.ndarray,
                              save_path: str = None, verbose: bool = False) -> pydicom.dataset.FileDataset:
    """
    Create a new DICOM file containing a segmentation mask,
    preserving metadata from the original DICOM image.

    Parameters
    ----------
    original_dcm : pydicom.dataset.FileDataset
        The original DICOM dataset (loaded with pydicom.dcmread).
    seg_array : np.ndarray
        Segmentation array (same shape as original_dcm.pixel_array).
        Typically a binary or labeled mask (uint8 or uint16).
    save_path : str, optional
        If provided, saves the new DICOM to this path.

    Returns
    -------
    new_dcm : pydicom.dataset.FileDataset
        The new DICOM dataset containing the segmentation.
    """
    # ---- Copy dataset ----
    new_dcm = original_dcm.copy()

    # ---- Convert segmentation array ----
    # ---- Replace pixel data ----
    if len(seg_array.shape) == 2:
        new_dcm.PixelData = seg_array.tobytes()
        new_dcm.Rows, new_dcm.Columns = seg_array.shape[0], seg_array.shape[1]
    else:
        new_dcm.PixelData = seg_array.tobytes()
        new_dcm.Rows, new_dcm.Columns, new_dcm.NumberOfFrames = seg_array.shape[1], seg_array.shape[2], seg_array.shape[0]

    # ---- Update identifying metadata ----
    new_dcm.SeriesDescription = original_dcm.SeriesDescription + " Segmentation Mask"
    new_dcm.SOPInstanceUID = generate_uid()
    new_dcm.SeriesInstanceUID = generate_uid()
    new_dcm.file_meta.MediaStorageSOPInstanceUID = new_dcm.SOPInstanceUID

    # ---- Adjust pixel representation ----
    if seg_array.dtype == np.uint8:
        new_dcm.BitsAllocated = 8
        new_dcm.BitsStored = 8
        new_dcm.HighBit = 7
    else:
        new_dcm.BitsAllocated = 16
        new_dcm.BitsStored = 16
        new_dcm.HighBit = 15
    new_dcm.PixelRepresentation = 0  # unsigned


    # ---- Save if requested ----
    if save_path:
        new_dcm.save_as(save_path)
        if verbose:
            print(f"✅ Segmentation DICOM saved to: {save_path}")

    return new_dcm
import pandas as pd

def calculate_ED_ES_and_metrics(file_path: str, output_path: str):
    # Load the CSV into a DataFrame
    df = pd.read_csv(file_path)
    
    # Initialize an empty list to store results
    results = []

    # Group by folder name
    grouped = df.groupby('folder_name')

    # Iterate through each group (folder_name)
    for folder_name, group in grouped:
        # Get the slice with the maximum and minimum total volume for ED and ES
        ED_slice = group.loc[group['volume_total'].idxmax()]
        ES_slice = group.loc[group['volume_total'].idxmin()]
        print("Processing folder: ", folder_name, "slice ED idx: ", ED_slice['slice_number'], "slice ES idx: ", ES_slice['slice_number'])

        # Calculate Ejection Fraction
        ejection_fraction_LV = round((ED_slice['volume_LV'] - ES_slice['volume_LV']) / ED_slice['volume_LV'] * 100, 2)

        # Calculate Myocardium Mass for ED and ES
        myocardium_mass_ED = round(ED_slice['volume_MYO'] * 1.05, 2)
        myocardium_mass_ES = round(ES_slice['volume_MYO'] * 1.05, 2)
        # Store results for the folder
        results.append({
            'folder_name': folder_name,
            'volume_LV_ED': ED_slice['volume_LV'],
            "slice_number_ED": ED_slice['slice_number'],
            'volume_LV_ES': ES_slice['volume_LV'],
            "slice_number_ES": ES_slice['slice_number'],
            'ejection_fraction_LV': ejection_fraction_LV,
            "volume_MYO_ED": ED_slice['volume_MYO'],
            'myocardium_mass_ED': myocardium_mass_ED,
            "volume_MYO_ES": ES_slice['volume_MYO'],
            'myocardium_mass_ES': myocardium_mass_ES
        })

    # Convert results into DataFrame
    result_df = pd.DataFrame(results)

    # Save the result to a new CSV file
    result_df.to_csv(output_path, index=False)
    print(f"✅ Results saved to: {output_path}")



In [ ]:
csv_file = f"../data/csv_files/result_mouse_retro_8sl_4NSA-40FA.csv"

f = open(csv_file, mode="w")
writer = csv.DictWriter(f, fieldnames=["folder_name", "slice_number", "volume_LV", "volume_MYO", "volume_total"])
writer.writeheader()
list_test_folder = natsorted(glob.glob("../data/M51xx/*"))
destination_folder = "../data/predicted_mouse_retro_8sl_4NSA-40FA"
os.makedirs(destination_folder, exist_ok=True)
count = 0

for folder in list_test_folder:
    list_test_image = natsorted(glob.glob(f"{folder}/*"))
    images = []
    folder_name = folder.split("/")[-1]
    slice_number = 0
    for image_path in list_test_image:
        count += 1
        if count < 15: 
            sample_dcm = dcmread(image_path)
            pixel_array = sample_dcm.pixel_array
            images.append(pixel_array)
            continue
        if count == 15:
            slice_number += 1
            sample_dcm = dcmread(image_path)
            pixel_array = sample_dcm.pixel_array
            images.append(pixel_array)
            image_array = np.stack(images, axis=-1)

            data = preprocess_data_mouse_retro(image_array)
            seg = predict_data_model(data, model_acdc, num_classes=4).astype(np.uint8)
            seg = np.transpose(seg, (2, 0, 1)) # 15, 256, 256
            # give label 1, 3, 4 to 0
            seg[seg == 1] = 0
            seg[seg == 3] = 1
            seg = seg.astype(np.uint8)

            volume_LV = round(make_volume(seg==1, sample_dcm.PixelSpacing), 2)
            volume_MYO = round(make_volume(seg==2, sample_dcm.PixelSpacing), 2)
            volume_total = volume_LV+volume_MYO
            images = []
            count = 0

            writer.writerow({"folder_name": folder_name, "slice_number": slice_number, "volume_LV": volume_LV, "volume_MYO": volume_MYO, "volume_total": volume_total})

            seg_dcm = create_segmentation_dicom(sample_dcm, seg, f"{destination_folder}/{folder_name}_{slice_number}_tiramisu_seg.dcm")


        # break
f.close()
#stack the result dim = -1
# Example usage


In [ ]:
calculate_ED_ES_and_metrics(csv_file, "../data/csv_files/result_mouse_retro_8sl_4NSA-40FA_EF.csv")


In [ ]:
list_test_folder = natsorted(glob.glob("../data/M51xx/*"))
destination_folder = "../data/predicted_M51xx"
os.makedirs(destination_folder, exist_ok=True)
count = 0

for folder in list_test_folder:
    list_test_image = natsorted(glob.glob(f"{folder}/*"))
    images = []
    folder_name = folder.split("/")[-1]
    # create folder name
    os.makedirs(f"{destination_folder}/{folder_name}", exist_ok=True)
    for image_path in list_test_image:

        sample_dcm = dcmread(image_path)
        pixel_array = sample_dcm.pixel_array
        images.append(pixel_array)
        image_array = np.stack(images, axis=-1)

    data = preprocess_data_mouse_retro(image_array)
    seg = predict_data_model(data, model_acdc, num_classes=4).astype(np.uint8)
    seg = np.transpose(seg, (2, 0, 1)) # 15, 256, 256
    # give label 1, 3, 4 to 0
    seg[seg == 1] = 0
    seg[seg == 3] = 1
    seg = seg.astype(np.uint8)

    for i in range(seg.shape[0]):
        seg_dcm = create_segmentation_dicom(sample_dcm, seg[i], f"{destination_folder}/{folder_name}/tiramisu_seg_{i+1}.dcm")





In [ ]:
# import canvas
from reportlab.pdfgen import canvas
# import blue color
from reportlab.lib.colors import *
from reportlab.lib.utils import ImageReader
import io
import argparse
import numpy as np
import matplotlib.pyplot as plt
import configparser
import time
import sys
sys.path.append("../src/")
sys.path.append("../")
from segment2d import *


# Processing folder:  M5151 slice ED idx:  6 slice ES idx:  1
# Processing folder:  M5152 slice ED idx:  6 slice ES idx:  2
# Processing folder:  M5156 slice ED idx:  5 slice ES idx:  1

def create_acdc_report(pdf_file,input_path, seg_path, csv_file):

    # --- Load data ---
    dcm = dcmread(input_path)
    seg = dcmread(seg_path).pixel_array
    folder_name = input_path.split("/")[-2]
    # load csv file
    df = pd.read_csv(csv_file)
    df = df[df['folder_name'] == folder_name]
    myocardium_volume_ED = df.iloc[0]['volume_MYO_ED']
    left_ventricle_volume_ED = df.iloc[0]['volume_LV_ED']
    myocardium_volume_ES = df.iloc[0]['volume_MYO_ES']
    left_ventricle_volume_ES = df.iloc[0]['volume_LV_ES']
    # calculate the ejection fraction
    lv_ef = df.iloc[0]['ejection_fraction_LV']
    myocardium_mass_ED = df.iloc[0]['myocardium_mass_ED']
    myocardium_mass_ES = df.iloc[0]['myocardium_mass_ES']
    # center crop the image and seg
    image = dcm.pixel_array
    w, h = image.shape[:2]
    image = image[w//2-64:w//2+64, h//2-64:h//2+64]
    seg = seg[w//2-64:w//2+64, h//2-64:h//2+64]


    # --- Create PDF canvas ---
    pagesize = (600, 600)
    c = canvas.Canvas(pdf_file, pagesize=pagesize)
    width, height = pagesize

    # --- Header ---
    c.drawImage('../figures/logo1.png', 5, height - 80, width=80, height=80)
    c.setFont("Helvetica-Bold", 30)
    # make it center horizontally
    c.drawCentredString(width / 2, height - 50, "PATIENT REPORT")

    # --- Patient info ---
    patient_dict = {"Case": folder_name,
    "Sex": dcm.PatientSex,
    "Age": dcm.PatientAge,
    "Series Description": dcm.SeriesDescription,
    "Slice ED": df.iloc[0]['slice_number_ED'],
    "Slice ES": df.iloc[0]['slice_number_ES'],
    "Manufacturer": dcm.Manufacturer,
    "Institution Name": dcm.InstitutionName,
    }
    infor_list = ["Case", "Sex", "Age"]
    infor_x = 50
    for infor in infor_list:
        c.setFont("Helvetica-Bold", 14)
        c.setFillColor(blue)
        c.drawString(infor_x, height - 100, f"{infor}")
        c.setFillColor(black)
        c.setFont("Helvetica", 12)
        c.drawString(infor_x, height - 120, str(patient_dict.get(infor, "Data not available")))
        infor_x += 160

    # --- Create combined figure with 2 subplots ---
    fig, ax = plt.subplots(1, 2, figsize=(5, 5))
    
    ax[0].imshow(image, cmap="gray")
    ax[1].imshow(image, cmap="gray", alpha=0.7)
    ax[1].imshow(seg, cmap="jet", alpha=0.5)
    ax[0].axis("off")
    ax[1].axis("off")
        # adjust the space between the images   
    plt.subplots_adjust(wspace=0.05)

    # Save figure to buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)

    # --- Draw combined image on PDF ---
    image_x = 80
    image_y = height - 280
    image_w = 400
    image_h = 150
    c.drawImage(ImageReader(buf), image_x, image_y, width=image_w, height=image_h)
    # delete the buffer
    del buf
    # --- Add legend to the right of the image ---
    legend_x = image_x + image_w + 10
    legend_y = image_y + image_h / 2 - 5

    c.setFont("Helvetica", 10)
    c.setFillColor(HexColor("#69ab69"))
    c.rect(legend_x, legend_y, 10, 10, fill=1, stroke=0)
    c.setFillColor(HexColor("#a36464"))
    c.rect(legend_x, legend_y - 20, 10, 10, fill=1, stroke=0)
    # c.setFillColor(HexColor("#00AEEF"))
    # c.rect(legend_x, legend_y - 40, 10, 10, fill=1, stroke=0)
    c.setFillColor(black)
    c.drawString(legend_x + 15, legend_y + 1, "Left Ventricle (LV)")
    c.drawString(legend_x + 15, legend_y - 19, "Myocardium (MYO)")
    # c.drawString(legend_x + 15, legend_y - 39, "Right Ventricle (RV)")


    # --- Clinical data section ---
    col_width = width / 2
    clinical_y = height - 300
    c.setFont("Helvetica-Bold", 14)
    c.setFillColor(red)
    c.drawCentredString(col_width / 2, clinical_y, "Clinical Data")
    clinical_y -= 20
    c.setFillColor(black)
    c.setFont("Helvetica", 12)
    list_infor_remove = ["Case", "Sex", "Age"]
    for key, value in patient_dict.items():
        if key not in list_infor_remove:
            c.drawString(50, clinical_y, f"{key}: {value}")
            clinical_y -= 20

    # --- Segmentation data section ---
    segment_y = height - 300
    c.setFont("Helvetica-Bold", 14)
    c.setFillColor(red)
    c.drawCentredString(col_width + (col_width / 2), segment_y, "Segmentation Data")
    segment_y -= 20
    c.setFont("Helvetica", 12)
    segment_data = {
        "Volume": {
            "LV ED": f"{left_ventricle_volume_ED} mL",
            "LV ES": f"{left_ventricle_volume_ES} mL",
            "MYO ED": f"{myocardium_volume_ED} mL",
            "MYO ES": f"{myocardium_volume_ES} mL",
        },
        "Ejection Fraction": {
            "LV": f"{lv_ef}%",
        },
        "Myocardium Mass": {
            "MYO Mass ED": f"{myocardium_mass_ED} g",
            "MYO Mass ES": f"{myocardium_mass_ES} g",
        }
    }
    c.setFillColor(black)
    for category, details in segment_data.items():
        c.drawString(col_width + 10, segment_y, f"{category}:")
        segment_y -= 20
        for sub_key, sub_value in details.items():
            c.drawString(col_width + 30, segment_y, f"{sub_key}: {sub_value}")
            segment_y -= 20
    # --- Footer note ---
    c.setFont("Helvetica-Oblique", 9)
    c.setFillColor(gray)
    c.drawCentredString(
        width / 2,
        10,
        "This report was automatically generated by a Deep Learning model "
        "using patient information and CMRI images."
    )
    print("Save report to: ", pdf_file)
    c.save()
input_path = "../data/M51xx/M5151/retro_8sl00085.dcm"
seg_path = "../data/predicted_M51xx/M5151/tiramisu_seg_85.dcm"
csv_file = "../data/csv_files/result_mouse_retro_8sl_4NSA-40FA_EF.csv"
create_acdc_report("../pdf_files/M5151_acdc_report.pdf", input_path, seg_path, csv_file)

In [ ]:
input_image= dcmread(input_path)
input_image 

In [ ]:
mouse_data_path = "../data/494_Pzac_M00457241_Ep3533_151022/Unnamed - 695116639/retro_8sl_4NSA40FA_901/*"

In [ ]:
# make widget to imshow the mouse data and the predicted mask
def imshow_mouse_data_and_predicted_mask(index):
    dcm = dcmread(mouse_path[index])
    seg = dcmread(seg_path)
    # plot the mouse data and the predicted mask
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(dcm.pixel_array)
    ax[1].imshow(seg.pixel_array[index])
    plt.show()
interact(imshow_mouse_data_and_predicted_mask, index=(0, len(mouse_path) - 1))

In [ ]:
# make widget to imshow the mouse data and the predicted mask
def imshow_mouse_data_and_predicted_mask(index):
    dcm = dcmread(mouse_data_path[index])
    seg = dcmread(seg_path[index])
    # plot the mouse data and the predicted mask
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(dcm.pixel_array)
    ax[1].imshow(seg.pixel_array)
    plt.show()
interact(imshow_mouse_data_and_predicted_mask, index=(0, len(mouse_data_path) - 1))


In [ ]:
image = sample_dcm.pixel_array
mask = sample_dcm.pixel_array

def plot_image_mask_z(image, mask, padded_image, padded_mask, z):
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(image[..., z], cmap="gray")
    ax[0].set_title("Image")
    ax[0].imshow(mask[..., z], cmap="jet", alpha=0.)
    ax[1].imshow(padded_image[..., z], cmap="gray", alpha=0.7)
    ax[1].set_title("Image predict")
    ax[1].imshow(padded_mask[..., z], cmap="jet", alpha=0.3)
    # off the axis
    ax[0].axis('off')
    ax[1].axis('off')


interact(lambda z: plot_image_mask_z(image, mask, image, mask, z), z=(0, image.shape[-1] - 1))

In [ ]:
mouse_data_path = "../data/494_Pzac_M00457241_Ep3533_151022/Unnamed - 695116639/retro_8sl_4NSA40FA_901/*"
mouse_data_path = glob.glob(mouse_data_path)
mouse_data_path = natsorted(mouse_data_path)

In [ ]:
mouse_data_path = "../data/M51xx/M5156/DICOM/*"
mouse_data_path = glob.glob(mouse_data_path)
mouse_data_path = natsorted(mouse_data_path)
# make widget to imshow the mouse data
def imshow_mouse_data(index):
    dcm = dcmread(mouse_data_path[index])
    plt.imshow(dcm.pixel_array)
    plt.show()
interact(imshow_mouse_data, index=(0, len(mouse_data_path) - 1))

